In [0]:
print("Running identity:")
display(spark.sql("SELECT current_user()"))

In [0]:
%sql
create table if not exists devdatabricksweather.bronze.bronze_table

In [0]:
dbutils.widgets.text("raw_path", "")
dbutils.widgets.text("cityName", "")
dbutils.widgets.text("fileName", "")
dbutils.widgets.text("bronze_table", "")
cityName = dbutils.widgets.get("cityName")
fileName = dbutils.widgets.get("fileName")
bronze_table = dbutils.widgets.get("bronze_table")
raw_path = dbutils.widgets.get("raw_path")
#file_name = "weather_data_2026_08_14"


df_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{raw_path}/raw/{cityName}/{fileName}")
)



In [0]:
from pyspark.sql import functions as F
df_bronze = (
    df_raw
    .withColumn("_ingest_file", F.lit(fileName))
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("city", F.lit(cityName))
)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("use catalog devdatabricksweather")
spark.sql("use database bronze")
(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable('bronze_table')
)

In [0]:
%sql
select * from devdatabricksweather.bronze.bronze_table

In [0]:
row_count = df_bronze.count()
display(row_count)